# Vision AI: train our own object detector
**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Internet → **On**.
Add data with **+ Add Input**: search `coco-2017-dataset` (by awsaf49), plus any YOLO-format datasets (Red Bull cans, phones, tools…). You can also list GitHub dataset repos in the cell below.

In [ ]:
!pip -q install 'transformers>=4.45' pyyaml scipy

In [ ]:
%%writefile merge_datasets.py
"""Merge many detection datasets (COCO-JSON and/or YOLO-txt format) into ONE
COCO-format dataset with a unified, contiguous label list.

Classes are matched by name (case-insensitive), so "Cup" from one dataset and
"cup" from another become the same class. Use --rename to fix synonyms,
e.g. --rename "cell phone=phone" --rename "mobile=phone".

Examples:
  python tools/merge_datasets.py --out merged \
      --coco /data/coco/annotations/instances_val2017.json:/data/coco/val2017 \
      --yolo /data/redbull_yolo            # folder containing data.yaml
"""
import argparse, json, os, glob
from PIL import Image
import yaml

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def load_coco(ann_path, img_dir):
    d = json.load(open(ann_path))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    by_img = {}
    for a in d["annotations"]:
        if a.get("iscrowd", 0):
            continue
        by_img.setdefault(a["image_id"], []).append((cats[a["category_id"]], a["bbox"]))
    for im in d["images"]:
        yield os.path.join(img_dir, im["file_name"]), im["width"], im["height"], by_img.get(im["id"], [])


def load_yolo(root):
    cfg_path = next(iter(glob.glob(os.path.join(root, "*.yaml"))), None)
    if cfg_path is None:
        raise SystemExit(f"No data.yaml found in {root}")
    cfg = yaml.safe_load(open(cfg_path))
    names = cfg["names"]
    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    for img_path in glob.glob(os.path.join(root, "**", "*"), recursive=True):
        if not img_path.lower().endswith(IMG_EXT):
            continue
        # YOLO convention: .../images/x.jpg  ->  .../labels/x.txt
        lbl = os.path.splitext(img_path.replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}"))[0] + ".txt"
        if not os.path.exists(lbl):
            continue
        w, h = Image.open(img_path).size
        objs = []
        for line in open(lbl):
            p = line.split()
            if len(p) < 5:
                continue
            c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
            objs.append((names[c], [(cx - bw / 2) * w, (cy - bh / 2) * h, bw * w, bh * h]))
        yield img_path, w, h, objs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True, help="output folder (writes annotations.json)")
    ap.add_argument("--coco", action="append", default=[], help="ANN_JSON:IMAGE_DIR (repeatable)")
    ap.add_argument("--yolo", action="append", default=[], help="YOLO dataset root (repeatable)")
    ap.add_argument("--rename", action="append", default=[], help="'old=new' class rename (repeatable)")
    ap.add_argument("--max-images-per-source", type=int, default=0, help="0 = all")
    args = ap.parse_args()

    rename = {k.strip().lower(): v.strip().lower() for k, v in (r.split("=", 1) for r in args.rename)}
    sources = [load_coco(*s.rsplit(":", 1)) for s in args.coco] + [load_yolo(r) for r in args.yolo]

    label2id, images, anns = {}, [], []
    for src in sources:
        for n, (path, w, h, objs) in enumerate(src):
            if args.max_images_per_source and n >= args.max_images_per_source:
                break
            img_id = len(images)
            images.append({"id": img_id, "file_name": os.path.abspath(path), "width": w, "height": h})
            for name, (x, y, bw, bh) in objs:
                name = rename.get(name.lower(), name.lower())
                if bw <= 1 or bh <= 1:
                    continue
                cid = label2id.setdefault(name, len(label2id))
                anns.append({"id": len(anns), "image_id": img_id, "category_id": cid,
                             "bbox": [x, y, bw, bh], "area": bw * bh, "iscrowd": 0})

    os.makedirs(args.out, exist_ok=True)
    out = {"images": images, "annotations": anns,
           "categories": [{"id": i, "name": n} for n, i in label2id.items()]}
    json.dump(out, open(os.path.join(args.out, "annotations.json"), "w"))
    print(f"Merged {len(images)} images, {len(anns)} boxes, {len(label2id)} classes -> {args.out}/annotations.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_detector.py
"""Fine-tune an RT-DETR object detector (Apache-2.0, OK to sell) on a merged
COCO-format dataset produced by tools/merge_datasets.py.

Run on Kaggle with a GPU:
  python train_detector.py --ann merged/annotations.json --out /kaggle/working/model --epochs 10
"""
import argparse, json, math, os, random
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

BASE = "PekingU/rtdetr_r50vd_coco_o365"  # pretrained on COCO + Objects365


class CocoDet(Dataset):
    def __init__(self, images, by_img, processor, train):
        self.images, self.by_img, self.proc, self.train = images, by_img, processor, train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        im = self.images[i]
        img = Image.open(im["file_name"]).convert("RGB")
        anns = [dict(a) for a in self.by_img.get(im["id"], [])]
        if self.train and random.random() < 0.5:  # horizontal flip
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            for a in anns:
                x, y, w, h = a["bbox"]
                a["bbox"] = [img.width - x - w, y, w, h]
        enc = self.proc(images=img, annotations={"image_id": im["id"], "annotations": anns}, return_tensors="pt")
        return enc["pixel_values"][0], enc["labels"][0]


def collate(batch):
    return {"pixel_values": torch.stack([b[0] for b in batch]), "labels": [b[1] for b in batch]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ann", required=True)
    ap.add_argument("--out", default="model")
    ap.add_argument("--base", default=BASE)
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--val-frac", type=float, default=0.05)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--max-steps", type=int, default=0, help="stop early (for smoke tests)")
    args = ap.parse_args()

    d = json.load(open(args.ann))
    id2label = {c["id"]: c["name"] for c in d["categories"]}
    by_img = {}
    for a in d["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)
    images = [im for im in d["images"] if im["id"] in by_img]
    random.seed(0); random.shuffle(images)
    n_val = max(1, int(len(images) * args.val_frac))
    val_imgs, train_imgs = images[:n_val], images[n_val:]
    print(f"{len(train_imgs)} train / {len(val_imgs)} val images, {len(id2label)} classes")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc = RTDetrImageProcessor.from_pretrained(args.base)
    model = RTDetrForObjectDetection.from_pretrained(
        args.base, id2label=id2label, label2id={v: k for k, v in id2label.items()},
        ignore_mismatched_sizes=True).to(device)

    mk = lambda ims, tr: DataLoader(CocoDet(ims, by_img, proc, tr), batch_size=args.batch, shuffle=tr,
                                    num_workers=args.workers, collate_fn=collate)
    train_dl, val_dl = mk(train_imgs, True), mk(val_imgs, False)

    backbone = [p for n, p in model.named_parameters() if "backbone" in n]
    rest = [p for n, p in model.named_parameters() if "backbone" not in n]
    opt = torch.optim.AdamW([{"params": backbone, "lr": args.lr / 10}, {"params": rest, "lr": args.lr}],
                            weight_decay=1e-4)
    total = args.epochs * len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 300) * 0.5 * (1 + math.cos(math.pi * s / total)))
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    step, best = 0, float("inf")
    for ep in range(args.epochs):
        model.train()
        for b in train_dl:
            labels = [{k: v.to(device) for k, v in l.items()} for l in b["labels"]]
            with torch.autocast(device_type=device, enabled=use_amp):
                loss = model(pixel_values=b["pixel_values"].to(device), labels=labels).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if step % 50 == 0 or step == 1:
                print(f"epoch {ep} step {step}/{total} loss {loss.item():.3f}", flush=True)
            if args.max_steps and step >= args.max_steps:
                break

        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for b in val_dl:
                labels = [{k: v.to(device) for k, v in l.items()} for l in b["labels"]]
                vl += model(pixel_values=b["pixel_values"].to(device), labels=labels).loss.item(); n += 1
        vl /= max(n, 1)
        print(f"== epoch {ep} val loss {vl:.3f}")
        if vl < best:
            best = vl
            model.save_pretrained(args.out); proc.save_pretrained(args.out)
            print(f"   saved best model -> {args.out}")
        if args.max_steps and step >= args.max_steps:
            break


if __name__ == "__main__":
    main()


## 1. Collect data from Kaggle inputs + GitHub

In [ ]:
import glob, os, subprocess

# Datasets hosted on GitHub (YOLO format with a data.yaml). Add repo URLs here.
GITHUB_DATASETS = [
    # "https://github.com/<user>/<yolo-dataset-repo>.git",
]
for url in GITHUB_DATASETS:
    subprocess.run(["git", "clone", "--depth", "1", url, "gh_" + url.rstrip("/").split("/")[-1].removesuffix(".git")])

COCO_IMAGES_PER_SOURCE = 20000   # raise for the full run, lower for a quick test

args = []
for ann in glob.glob("/kaggle/input/**/instances_train2017.json", recursive=True):
    img_dir = glob.glob(os.path.join(os.path.dirname(os.path.dirname(ann)), "**", "train2017"), recursive=True)
    img_dir = [d for d in img_dir if os.path.isdir(d)]
    if img_dir:
        args += ["--coco", f"{ann}:{img_dir[0]}"]
for y in glob.glob("/kaggle/input/**/data.yaml", recursive=True) + glob.glob("gh_*/**/data.yaml", recursive=True):
    args += ["--yolo", os.path.dirname(y)]
print("Sources:", args)
assert args, "No datasets found: add an input dataset or a GitHub repo above"

renames = ["cell phone=phone", "mobile phone=phone", "redbull=red bull can"]
cmd = ["python", "merge_datasets.py", "--out", "merged", "--max-images-per-source", str(COCO_IMAGES_PER_SOURCE)]
cmd += args + sum([["--rename", r] for r in renames], [])
subprocess.run(cmd, check=True)

## 2. Train (≈ 1–2 h per 10k images per epoch on a T4)

In [ ]:
!python train_detector.py --ann merged/annotations.json --out /kaggle/working/model --epochs 10 --batch 8

## 3. Test on a picture

In [ ]:
import torch, json, random
from PIL import Image, ImageDraw
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
proc = RTDetrImageProcessor.from_pretrained("/kaggle/working/model")
model = RTDetrForObjectDetection.from_pretrained("/kaggle/working/model").cuda().eval()
im = random.choice(json.load(open("merged/annotations.json"))["images"])
img = Image.open(im["file_name"]).convert("RGB")
with torch.no_grad():
    out = model(**proc(images=img, return_tensors="pt").to("cuda"))
res = proc.post_process_object_detection(out, target_sizes=[img.size[::-1]], threshold=0.5)[0]
d = ImageDraw.Draw(img)
for s, l, b in zip(res["scores"], res["labels"], res["boxes"]):
    d.rectangle(b.tolist(), outline="lime", width=3)
    d.text((b[0] + 4, b[1] + 4), f"{model.config.id2label[l.item()]} {s:.0%}", fill="lime")
    print("This is a", model.config.id2label[l.item()], f"({s:.0%})")
img

## 4. Download the model
Zip it, then grab `model.zip` from the **Output** panel and copy the folder to your laptop for `python webcam_demo.py --model model`.

In [ ]:
!cd /kaggle/working && zip -qr model.zip model && ls -lh model.zip